<a id="top"></a>

# Lab L1004: build the agent loop (termination)

```yaml
title: "Lab L1004: build the agent loop (termination)"
keywords: agent loop, run_loop, stub model, bind_tools, tool_calls, ToolMessage, natural termination, max_steps cap, message history invariant, offline, lab
estimated duration: 40
```

> **Lesson:** L10. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L10/objectives.md). Solutions: `L1004_lab_solutions.ipynb`. **Pure Python — no API key needed.** You build the loop against a *scripted stub model*, so it runs the same way every time (the same offline approach as the [L1003 demo](L1003_lecture.ipynb)).
>
> **After this lab you can:** write a model→tool→model loop that maintains the message-history invariant, terminates naturally when the model stops calling tools, and halts on a `max_steps` cap when it doesn't.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Detect natural termination](#2-problem-1--detect-natural-termination)
- [3. Problem 2 — Write run_loop](#3-problem-2--write-run_loop)
- [4. Problem 3 — Drive it: natural termination](#4-problem-3--drive-it-natural-termination)
- [5. Problem 4 — Drive it: the max_steps cap catches a runaway](#5-problem-4--drive-it-the-max_steps-cap-catches-a-runaway)
- [6. Problem 5 — Two tool calls in one response (written)](#6-problem-5--two-tool-calls-in-one-response-written)
- [7. Self-check](#7-self-check)

## 1. Setup

All given: the `calculator` + `lookup` tools and a dispatch table; a scripted **`FakeModel`** (from `common`) that mimics a LangChain chat model — `.bind_tools(...)` then `.invoke(messages)` returns the next scripted `AIMessage` (its `.tool_calls` say which tools it wants run); a `dispatch` helper that runs one tool and returns a `ToolMessage`; and the `RunResult` dataclass your loop will return. You write **only** the loop.

In [1]:
from collections.abc import Callable
from dataclasses import dataclass

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.messages.tool import ToolCall

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# Dispatch table: tool NAME -> the function that runs it. The same functions,
# passed to model.bind_tools([...]), also ARE the schema a real model sees.
TOOLS: dict[str, Callable[..., str]] = {"calculator": calculator, "lookup": lookup}

In [2]:
def dispatch(tools: dict[str, Callable[..., str]], call: ToolCall) -> ToolMessage:
    """Run one requested tool and return a ToolMessage (failures become status='error')."""
    fn = tools.get(call["name"])
    if fn is None:
        return ToolMessage(
            content=f"no such tool {call['name']!r}",
            tool_call_id=call["id"],
            status="error",
        )
    try:
        output = fn(**call["args"])
        return ToolMessage(content=output, tool_call_id=call["id"], status="success")
    except Exception as exc:
        return ToolMessage(content=repr(exc), tool_call_id=call["id"], status="error")

In [3]:
@dataclass
class RunResult:
    """The answer, the number of model calls, and WHY the loop stopped."""

    final_text: str
    iterations: int
    termination: str  # "natural" or "max_steps"

[↑ Back to top](#top)

## 2. Problem 1 — Detect natural termination

Warm-up before the full loop. Write `is_done(reply)` that returns `True` when a model reply has **no tool calls** (only text) — that is *natural termination*, the one signal that means 'the model thinks it's done.' A reply is done iff `reply.tool_calls` is empty.

In [4]:
def is_done(reply: AIMessage) -> bool:
    """Natural termination: the reply carries no tool calls."""
    return not reply.tool_calls


print(is_done(text_reply("all done")))  # expect True
print(is_done(tool_reply(tool_call("c1", "lookup", {"city": "Paris"}))))  # expect False

True
False


[↑ Back to top](#top)

## 3. Problem 2 — Write run_loop

Now the whole loop. `run_loop(model, tools, user_msg, max_steps)` should first `model.bind_tools(list(tools.values()))`, then for up to `max_steps` iterations:

1. `.invoke(messages)` → an `AIMessage` reply.
2. If it has **no** `tool_calls`, return `RunResult(reply.text, step, "natural")`.
3. Otherwise — **Rule 1, the message-history invariant** — append the assistant reply (`messages.append(reply)`), then run **every** `reply.tool_calls` via `dispatch(tools, call)` and append **one `ToolMessage` per call** before looping.
4. If the loop runs out of steps, return `RunResult("", max_steps, "max_steps")`.

Add a `print()` per iteration (iteration number + the tool calls) — your minimum-viable trace for L11.

In [5]:
def run_loop(
    model: object,  # any bind_tools-capable chat model (FakeModel here)
    tools: dict[str, Callable[..., str]],
    user_msg: str,
    max_steps: int,
) -> RunResult:
    bound = model.bind_tools(list(tools.values()))
    messages: list[BaseMessage] = [HumanMessage(content=user_msg)]
    for step in range(1, max_steps + 1):
        reply = bound.invoke(messages)
        if not reply.tool_calls:
            print(f"[step {step}] natural termination")
            return RunResult(reply.text, step, "natural")
        # Rule 1: record the assistant turn, then answer EVERY tool call with a
        # matching ToolMessage before the next model call.
        messages.append(reply)
        for call in reply.tool_calls:
            print(f"[step {step}] tool call: {call['name']}({call['args']})")
            messages.append(dispatch(tools, call))
    print(f"[stop] max_steps={max_steps} hit")
    return RunResult("", max_steps, "max_steps")

[↑ Back to top](#top)

## 4. Problem 3 — Drive it: natural termination

Script a stub that calls `calculator`, then `lookup`, then answers in plain text. Run `run_loop` with `max_steps=10` and confirm it terminates **naturally** in 3 iterations.

In [6]:
happy_script = [
    tool_reply(tool_call("c1", "calculator", {"expression": "36 * 1"})),
    tool_reply(tool_call("c2", "lookup", {"city": "Lagos"})),
    text_reply("Lagos has about 15,000,000 people."),
]
model = FakeModel(happy_script)
result = run_loop(model, TOOLS, "Population of Lagos?", max_steps=10)
print("\nfinal_text :", result.final_text)
print("iterations :", result.iterations)
print("termination:", result.termination)
assert result.termination == "natural" and result.iterations == 3

[step 1] tool call: calculator({'expression': '36 * 1'})
[step 2] tool call: lookup({'city': 'Lagos'})
[step 3] natural termination

final_text : Lagos has about 15,000,000 people.
iterations : 3
termination: natural


[↑ Back to top](#top)

## 5. Problem 4 — Drive it: the max_steps cap catches a runaway

Script a stub that **never stops** — one `lookup` call that the `FakeModel` will repeat forever. Run `run_loop` with `max_steps=5` and confirm the **cap** fires (termination `"max_steps"`, 5 iterations). This is the safety net: a loop with no cap is broken, not minimal.

In [7]:
runaway_script = [tool_reply(tool_call("r1", "lookup", {"city": "Tokyo"}))]
model = FakeModel(runaway_script)
result = run_loop(model, TOOLS, "Keep checking Tokyo.", max_steps=5)
print("\ntermination:", result.termination, "  <- the cap fired, not the model")
assert result.termination == "max_steps" and result.iterations == 5

[step 1] tool call: lookup({'city': 'Tokyo'})
[step 2] tool call: lookup({'city': 'Tokyo'})
[step 3] tool call: lookup({'city': 'Tokyo'})
[step 4] tool call: lookup({'city': 'Tokyo'})
[step 5] tool call: lookup({'city': 'Tokyo'})
[stop] max_steps=5 hit

termination: max_steps   <- the cap fired, not the model


[↑ Back to top](#top)

## 6. Problem 5 — Multiple tool calls in one reply (written)

A single model reply can request **more than one** tool call (its `.tool_calls` holds several). In a sentence or two: what must your loop do with all of them *before* it calls the model again, and why? (Hint: the message-history invariant — Rule 1.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1004_lab_solutions.ipynb`. Done when: `is_done` returns `True`/`False` correctly; the happy run terminates `"natural"` in 3 iterations; the runaway run terminates `"max_steps"` in 5; and you can say what the loop does with multiple tool calls (run all of them, append one `ToolMessage` per call before the next model call).

[↑ Back to top](#top)